In [1]:
from pathlib import Path
import shutil
import random
import logging

In [2]:
LOG_FILE = Path("../logs/app.log")

logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger(__name__)

logger.info("Validation Split Creation Started")

In [3]:
DATA_DIR = Path("../data/processed/chest_xray")

TRAIN_DIR = DATA_DIR / "train"

NEW_TRAIN_DIR = DATA_DIR / "new_train"
NEW_VAL_DIR = DATA_DIR / "new_val"

classes = [
    "NORMAL",
    "BACTERIAL_PNEUMONIA",
    "VIRAL_PNEUMONIA"
]

In [4]:
for cls in classes:

    (NEW_TRAIN_DIR / cls).mkdir(
        parents=True,
        exist_ok=True
    )

    (NEW_VAL_DIR / cls).mkdir(
        parents=True,
        exist_ok=True
    )

print("✅ Folder structure created")

✅ Folder structure created


In [5]:
random.seed(42)

for cls in classes:

    class_dir = TRAIN_DIR / cls

    images = list(class_dir.glob("*"))

    random.shuffle(images)

    split_idx = int(len(images) * 0.9)

    train_images = images[:split_idx]
    val_images = images[split_idx:]

    for img in train_images:

        shutil.copy(
            img,
            NEW_TRAIN_DIR / cls / img.name
        )

    for img in val_images:

        shutil.copy(
            img,
            NEW_VAL_DIR / cls / img.name
        )

    logger.info(
        f"{cls}: "
        f"Train={len(train_images)}, "
        f"Val={len(val_images)}"
    )

print("✅ Stratified Split Completed")

✅ Stratified Split Completed


In [6]:
def count_images(folder):
    return len(list(folder.glob("*")))


print("\nNEW TRAIN")
print("-" * 40)

for cls in classes:

    print(
        f"{cls}:",
        count_images(NEW_TRAIN_DIR / cls)
    )

print("\nNEW VALIDATION")
print("-" * 40)

for cls in classes:

    print(
        f"{cls}:",
        count_images(NEW_VAL_DIR / cls)
    )


NEW TRAIN
----------------------------------------
NORMAL: 1206
BACTERIAL_PNEUMONIA: 2277
VIRAL_PNEUMONIA: 1210

NEW VALIDATION
----------------------------------------
NORMAL: 135
BACTERIAL_PNEUMONIA: 253
VIRAL_PNEUMONIA: 135


In [7]:
logger.info("Validation Split Creation Completed Successfully")

print("✅ New Train and Validation Sets Ready")

✅ New Train and Validation Sets Ready


In [8]:
import torch

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU")

CUDA Available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [9]:
from PIL import Image
from pathlib import Path

train_dir = Path("../data/processed/chest_xray/new_train")

for cls in train_dir.iterdir():
    if cls.is_dir():
        img = next(cls.glob("*"))
        image = Image.open(img)

        print(
            cls.name,
            image.mode,
            image.size
        )

BACTERIAL_PNEUMONIA L (1152, 760)
NORMAL L (2090, 1858)
VIRAL_PNEUMONIA L (1072, 768)


We're checking:

RGB vs Grayscale
Image dimensions consistency

Because chest X-rays are often grayscale, but PyTorch models typically expect 3 channels.

## Save dataset statistics:

In [10]:
import pandas as pd
from pathlib import Path

ARTIFACTS_DIR = Path("../artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

dataset_stats = pd.DataFrame({
    "Class": [
        "NORMAL",
        "BACTERIAL_PNEUMONIA",
        "VIRAL_PNEUMONIA"
    ],
    "Train_Count": [
        1206,
        2277,
        1210
    ],
    "Validation_Count": [
        135,
        253,
        135
    ],
    "Test_Count": [
        234,
        242,
        148
    ]
})

dataset_stats.to_csv(
    ARTIFACTS_DIR / "dataset_stats.csv",
    index=False
)

print("✅ dataset_stats.csv saved")
display(dataset_stats)

✅ dataset_stats.csv saved


,Class,Train_Count,Validation_Count,Test_Count
0,NORMAL,1206,135,234
1,BACTERIAL_PNEUMONIA,2277,253,242
2,VIRAL_PNEUMONIA,1210,135,148
